# MLP Classification - نسخة المحاضر

هذا الدفتر مخصص للمحاضر، مع شرح تفصيلي لكل خطوة: ماذا نفعل، ولماذا، وكيف نفسر النتائج للطلاب.

## الهدف التعليمي
تصنيف ثنائي: هل المستخدم اشترى المنتج (Purchased) بناءً على **Age** و **EstimatedSalary**.
نبني MLP بـ Keras ونقيّمه بـ confusion matrix.

## خطة الشرح
1. استيراد المكتبات
2. قراءة البيانات
3. تجهيز X و y
4. تقسيم البيانات
5. Feature scaling
6. بناء MLP
7. compile + fit
8. التقييم والتنبؤ
9. Confusion matrix + scatter plot


## الخطوة 1: استيراد المكتبات

- **sklearn.metrics**: confusion matrix لقياس الأخطاء
- **tensorflow.keras**: Sequential + Dense لبناء MLP


### أولاً: استيراد المكتبات البرمجية المطلوبة (Import Libraries)
نقوم في هذه الخطوة باستيراد الأدوات والمكتبات اللازمة لمعالجة البيانات، بناء وتدريب النموذج، وتقييم النتائج:

- **`numpy` (المستوردة كـ `np`):** للعمليات الحسابية والتعامل مع المصفوفات الرياضية.
- **`pandas` (المستوردة كـ `pd`):** لقراءة البيانات وإدارة الجداول البرمجية (DataFrames).
- **`matplotlib.pyplot` (المستوردة كـ `plt`):** للرسم البياني وتصور البيانات بصرياً.
- **`train_test_split`:** لتقسيم البيانات إلى مجموعة تدريب ومجموعة اختبار بشكل عشوائي ومنظم.
- **`StandardScaler`:** لتقييس وتوحيد نطاق الخصائص (Feature Scaling) ليكون المتوسط صفر والانحراف المعياري واحد.
- **`tensorflow / keras`:** لبناء وتدريب الشبكات العصبية الاصطناعية ونماذج التعلم العميق.


In [ ]:
# الخطوة 1) استيراد المكتبات
# pip install tensorflow -q  # فعّل في Colab إذا لزم
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


## الخطوة 2: قراءة البيانات

مجموعة **Social Network Ads**: إعلانات شبكات اجتماعية.
نستخدم عمودي Age (العمر) و EstimatedSalary (الراتب) كميزات.


### أولاً: استيراد المكتبات البرمجية المطلوبة (Import Libraries)
نقوم في هذه الخطوة باستيراد الأدوات والمكتبات اللازمة لمعالجة البيانات، بناء وتدريب النموذج، وتقييم النتائج:




In [ ]:
# Step 2) قراءة البيانات / Load dataset
import os
import urllib.request

filename = 'Social_Network_Ads.csv'
if not os.path.exists(filename):
    url = 'https://raw.githubusercontent.com/iksasa15/AI-ML/main/code/15-%20Deep%20Learning/2-%20MLP%20Classification/Social_Network_Ads.csv'
    urllib.request.urlretrieve(url, filename)

dataset = pd.read_csv(filename)
dataset.head()


## الخطوة 3: تجهيز X و y

- **X**: الأعمدة 2 و 3 (Age, EstimatedSalary)
- **y**: العمود 4 (Purchased: 0 = لم يشتري، 1 = اشترى)


### سابعاً: تحديد المتغيرات المستقلة والتابعة (Features and Target)
نقوم بفصل البيانات المدخلة:
- **المتغيرات المستقلة ($X$):** الميزات والخصائص التي يستخدمها النموذج للتعلم والتنبؤ.
- **المتغير التابع ($y$):** الهدف أو المخرج الذي نريد من النموذج أن يتعلم توقعه.


In [ ]:
# الخطوة 3) تجهيز X و y
X = dataset.iloc[:, [2, 3]].values  # Age, EstimatedSalary
y = dataset.iloc[:, 4].values       # Purchased


## الخطوة 4: تقسيم البيانات

80% تدريب / 20% اختبار.


### ثامناً: تقسيم البيانات إلى مجموعتي تدريب واختبار (Train/Test Split)
نقسم البيانات بنسبة 20% لمجموعة الاختبار وبقية البيانات لمجموعة التدريب:
- **بيانات التدريب (Training Set):** لتعليم النموذج وضبط أوزانه ومعاملاته.
- **بيانات الاختبار (Test Set):** لتقييم النموذج واختبار قدرته على التنبؤ ببيانات جديدة كلياً.


In [ ]:
# الخطوة 4) تقسيم البيانات
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)


## الخطوة 5: Feature Scaling

Age بالسنوات والراتب بالآلاف — مقاييس مختلفة جداً.
StandardScaler يوحّد المقياس لتسريع التدريب واستقراره.


### تاسعاً: تقييس الخصائص (Feature Scaling)
نقوم بعملية التقييس أو المعايرة للبيانات:
- نستخدم `fit_transform` على مجموعة التدريب ليتعلم المتوسط والانحراف المعياري ويطبق التحويل.
- نستخدم `transform` فقط على مجموعة الاختبار لمنع تسرب البيانات (Data Leakage).
- هذه الخطوة ضرورية جداً للخوارزميات الحساسة للمقاييس مثل متجهات الدعم (SVM)، الجار الأقرب (KNN)، والشبكات العصبية.


In [ ]:
# الخطوة 5) Feature scaling
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)


## الخطوة 6: بناء MLP

البنية: **64 → 32 → 1**
- طبقتان مخفيتان ReLU: تعلم تمثيلات غير خطية
- طبقة إخراج sigmoid: احتمال الشراء (0–1)

`model.summary()` يعرض عدد المعاملات (params) لكل طبقة.


### عاشراً: بناء وتدريب نموذج الشبكة العصبية الاصطناعية (Neural Network)
1. نقوم بإنشاء كائن من النموذج بالمعاملات المناسبة.
2. نستخدم الدالة `.fit(X_train, y_train)` لتدريب النموذج على بيانات التدريب لكي يتعلم العلاقات والأنماط.


In [ ]:
# الخطوة 6) بناء MLP
model = Sequential([
    Dense(units=64, activation='relu', input_shape=(2,)),
    Dense(units=32, activation='relu'),
    Dense(units=1, activation='sigmoid')
])
model.summary()


## الخطوة 7: compile + fit

- **loss=binary_crossentropy**: مناسب للتصنيف الثنائي
- **optimizer=adam**: تحديث تكيفي للأوزان
- **validation_split=0.2**: 20% من التدريب للتحقق أثناء التدريب
- **epochs=50**: عدد مرات المرور على كل البيانات


### خطوة: الخطوة 7) compile + fit
نقوم بتشغيل هذا الجزء من الكود لتنفيذ العمليات البرمجية الموضحة في التعليقات أعلاه لتجهيز البيانات أو تهيئة النموذج.


In [ ]:
# الخطوة 7) compile + fit
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=50,
    validation_split=0.2,
    verbose=1
)


## الخطوة 8: التقييم والتنبؤ

نقيّم على **مجموعة الاختبار** التي لم يرها النموذج.
العتبة 0.5: إذا التنبؤ > 0.5 نصنّف كـ 1 (اشترى).


### الحادي عشر: التنبؤ بقيم مجموعة الاختبار (Make Predictions)
نستخدم النموذج المدرب للتنبؤ بالنتائج للمدخلات الموجودة في مجموعة الاختبار للتأكد من قدرة النموذج على التعميم على بيانات جديدة لم يتدرب عليها من قبل.


In [ ]:
# الخطوة 8) التقييم والتنبؤ
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test loss: {loss:.4f}')
print(f'Test accuracy: {accuracy:.2%}')

y_pred = (model.predict(X_test, verbose=0) > 0.5).astype(int).flatten()


## الخطوة 9: Confusion Matrix + Scatter

**Confusion Matrix** (مصفوفة الالتباس):
- TP: تنبأ 1 وكان 1 (صحيح إيجابي)
- TN: تنبأ 0 وكان 0 (صحيح سلبي)
- FP: تنبأ 1 وكان 0 (خطأ إيجابي)
- FN: تنبأ 0 وكان 1 (خطأ سلبي)

Scatter plot يوضح توزيع الفئات بعد التحجيم.


### الثاني عشر: تقييم أداء النموذج (Evaluation Metrics)
نقوم بحساب عدة مقاييس إحصائية لتقييم كفاءة ودقة النموذج المستعمل:

- **مصفوفة الارتباك (Confusion Matrix):** جدول يوضح التوقعات الصحيحة والخاطئة لكل فئة بالتفصيل.


In [ ]:
# الخطوة 9) Confusion matrix + scatter
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm).plot(cmap='Blues')
plt.title('Confusion Matrix - MLP')
plt.show()

plt.figure(figsize=(6, 4))
plt.scatter(X_test[y_test == 0, 0], X_test[y_test == 0, 1], c='blue', label='Not Purchased', alpha=0.7)
plt.scatter(X_test[y_test == 1, 0], X_test[y_test == 1, 1], c='red', label='Purchased', alpha=0.7)
plt.xlabel('Age (scaled)')
plt.ylabel('Estimated Salary (scaled)')
plt.title('Test Set (scaled features)')
plt.legend()
plt.show()
